# F1 Strategy Dataset | Pit Stop Prediction

### **Synthetic Dataset Description**
The dataset for this competition (both train and test) was inspired by F1 strategy dataset. Feature distributions are close to, but not exactly the same, as the original, and we intentionally remove Normalized_TyreLife which makes the prediction trivial. Feel free to use the original dataset as part of this competition, both to explore differences as well as to see whether incorporating the original in training improves model performance.

* Files
    * train.csv - the training set, with PitNextLap as target
    * test.csv - the test set, used to predict the likelihood for PitNextLap
    * sample_submission.csv - a sample submission file in the correct format

---

### **Original Dataset Description**

##### Overview
This dataset provides a **lap‑level view of Formula 1 races**, designed for race strategy analysis and machine learning applications.  
It transforms raw telemetry data into a structured format with engineered features that capture **tire degradation, race progression, and driver performance dynamics**.

##### Features Included
- Lap‑by‑lap race data for each driver  
- Tire compound and tire life tracking  
- Lap time and degradation metrics  
- Position changes across laps  
- Race progress indicators  
- Pit stop detection  
- **Target variable:** `PitNextLap` (predict whether a driver will pit next lap)

##### Use Cases
- Predicting pit stop decisions  
- Modeling race strategies  
- Analyzing tire degradation patterns  
- Driver performance and consistency analysis  
- Time‑series and classification tasks  

##### Data Processing
Built using **FastF1** with custom feature engineering:
- Lap time deltas  
- Cumulative degradation  
- Normalized tire life  
- Race progress metrics  

##### Notes
- Aggregated from multiple races (multi‑race dataset)  
- Missing or unreliable entries cleaned  
- Suitable for both beginners and advanced ML workflows  

##### Future Updates
- Additional seasons and races  
- Advanced strategy features (undercut/overcut modeling)  
- Weather and track condition integration  


###### Evaluation: Submissions are evaluated on area under the ROC curve between the predicted probability and the observed target.

In [9]:
# IMPAORT REQUIRED MODULES

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# READING THE DATASETS

train_df_synth = pd.read_csv(r'D:\Codes\Artificial_Intelligence\Machine_Learning\Machine-Learning-With-Scikit-Learn\Projects\Datasets\predicting-f1-pit-stops\Syntetic Dataset\train.csv')

test_df = pd.read_csv(r'D:\Codes\Artificial_Intelligence\Machine_Learning\Machine-Learning-With-Scikit-Learn\Projects\Datasets\predicting-f1-pit-stops\Syntetic Dataset\test.csv')

train_df_orig = pd.read_csv(r'D:\Codes\Artificial_Intelligence\Machine_Learning\Machine-Learning-With-Scikit-Learn\Projects\Datasets\predicting-f1-pit-stops\Original Dataset\f1_strategy_dataset_v4.csv')

In [3]:
train_df_synth.head()

,id,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap
0,0,D109,HARD,Canadian Grand Prix,2022,0,50,2,39.0,8,78.491,-7.564,21.019,0.714286,5.0,1.0
1,1,D086,HARD,Dutch Grand Prix,2025,1,27,2,7.0,4,75.095,-32.617,-223.207,0.346154,-3.0,0.0
2,2,ZON,HARD,Austrian Grand Prix,2022,0,59,3,22.0,13,70.945,-7.540,-100.529,0.819444,3.0,1.0
3,3,SPE,MEDIUM,Pre-Season Testing,2023,0,2,1,2.0,7,94.361,-7.324,-7.324,0.076923,0.0,0.0
4,4,D019,HARD,Azerbaijan Grand Prix,2022,1,26,3,6.0,2,107.878,8.965,-14.139,0.361111,3.0,0.0


In [4]:
train_df_orig.head()

,Driver,LapNumber,Compound,Stint,TyreLife,Position,LapTime (s),Race,Year,LapTime_Delta,Cumulative_Degradation,PitStop,PitNextLap,RaceProgress,Normalized_TyreLife,Position_Change
0,ALB,1,MEDIUM,1,2.0,17,100.625,Abu Dhabi Grand Prix,2023,0.000,0.000,0,0,0.017241,0.117647,0.0
1,ALB,2,MEDIUM,1,3.0,18,93.560,Abu Dhabi Grand Prix,2023,-7.065,-7.065,0,0,0.034483,0.176471,-1.0
2,ALB,3,MEDIUM,1,4.0,18,91.768,Abu Dhabi Grand Prix,2023,-1.792,-8.857,0,0,0.051724,0.235294,0.0
3,ALB,4,MEDIUM,1,5.0,18,91.591,Abu Dhabi Grand Prix,2023,-0.177,-9.034,0,0,0.068966,0.294118,0.0
4,ALB,5,MEDIUM,1,6.0,18,91.422,Abu Dhabi Grand Prix,2023,-0.169,-9.203,0,0,0.086207,0.352941,0.0


In [5]:
print(f'Syntheic Dataset columns:\n{train_df_synth.columns}\n')
print(f'Original Dataset columns:\n{train_df_orig.columns}')

Syntheic Dataset columns:
Index(['id', 'Driver', 'Compound', 'Race', 'Year', 'PitStop', 'LapNumber',
       'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta',
       'Cumulative_Degradation', 'RaceProgress', 'Position_Change',
       'PitNextLap'],
      dtype='object')

Original Dataset columns:
Index(['Driver', 'LapNumber', 'Compound', 'Stint', 'TyreLife', 'Position',
       'LapTime (s)', 'Race', 'Year', 'LapTime_Delta',
       'Cumulative_Degradation', 'PitStop', 'PitNextLap', 'RaceProgress',
       'Normalized_TyreLife', 'Position_Change'],
      dtype='object')


There is only one change is the original and synthetic dataset i.e. origial have a Normalized_TyreLife column but in the synthetic it has been removed.

##### Description of the features what do they show or mean:
### F1 Pit Stop Dataset Features Explained
* **Driver**: Name/identifier of the Formula 1 driver for the lap record. Helps group laps by driver and compare performance.
* **Compound**: Type of tire used (Soft, Medium, Hard, etc.). Critical for analyzing tire degradation and pit stop strategy.
* **Race**: The specific race event (e.g., Monaco GP, Italian GP). Useful for multi‑race comparisons and contextual analysis.
* **Year**: The season year of the race. Allows temporal analysis across seasons.
* **PitStop**: Indicates whether a pit stop occurred on that lap. Binary flag (Yes/No) or count of pit stops.
* **LapNumber**: Sequential lap number in the race. Provides time‑series ordering for race progression.
* **Stint**: Segment of laps between pit stops. Shows how long a driver runs on a given tire set.
* **TyreLife**: Number of laps completed on the current tire set. Key metric for tire wear and degradation modeling.
* **Position**: Driver’s race position at the end of the lap. Tracks competitiveness and race dynamics.
* **LapTime (s)**: Recorded lap time in seconds. Core performance measure per lap.
* **LapTime_Delta**: Difference in lap time compared to previous lap. Highlights performance changes, degradation, or improvements.
* **Cumulative_Degradation**: Aggregated measure of tire wear over laps. Useful for modeling tire performance decay.
* **RaceProgress**: Normalized indicator of how far into the race the lap is (e.g., % completed). Helps align laps across different races.
* **Position_Change**: Change in driver’s position compared to the previous lap. Captures overtakes, pit stop impacts, or race incidents.
* **PitNextLap**: **Target variable**: predicts whether the driver will pit on the next lap.Central to the competition’s machine learning task.


In [6]:
train_df_synth.describe()

,id,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap
count,439140.000000,439140.000000,439140.000000,439140.000000,439140.000000,439140.000000,439140.000000,439140.000000,439140.000000,439140.000000,439140.000000,439140.000000,439140.000000
mean,219569.500000,2023.523544,0.136118,23.105909,1.789113,14.158231,9.630339,90.948735,-3.770040,-25.721759,0.337661,0.101542,0.198982
std,126768.942943,1.024930,0.342915,16.958261,0.950194,9.801338,5.278770,19.772769,43.945759,54.766573,0.253277,4.006765,0.399235
min,0.000000,2022.000000,0.000000,1.000000,1.000000,1.000000,1.000000,67.694000,-2403.895000,-274.564000,0.012821,-18.000000,0.000000
25%,109784.750000,2023.000000,0.000000,9.000000,1.000000,6.000000,5.000000,82.621000,-8.884000,-46.566250,0.129870,-1.000000,0.000000
50%,219569.500000,2024.000000,0.000000,19.000000,2.000000,12.000000,10.000000,90.521000,-0.295000,-20.994000,0.269231,0.000000,0.000000
75%,329354.250000,2024.000000,0.000000,36.000000,2.000000,20.000000,14.000000,98.471000,0.115000,-6.199000,0.513158,2.000000,0.000000
max,439139.000000,2025.000000,1.000000,78.000000,8.000000,77.000000,20.000000,2507.607000,2423.932000,2412.026000,1.000000,18.000000,1.000000


In [7]:
train_df_orig.describe()

,LapNumber,Stint,TyreLife,Position,LapTime (s),Year,LapTime_Delta,Cumulative_Degradation,PitStop,PitNextLap,RaceProgress,Normalized_TyreLife,Position_Change
count,101371.000000,101371.000000,101371.000000,101371.000000,101371.000000,101371.000000,101371.000000,101371.000000,101371.000000,101371.000000,101371.000000,101371.000000,101371.000000
mean,30.444841,2.046394,14.549339,9.759132,92.587188,2023.589685,-0.203891,-29.550051,0.251581,0.254797,0.432618,0.386521,-0.004636
std,18.146942,0.948797,10.313385,5.406456,33.231414,1.098518,45.344910,70.235759,0.433924,0.435749,0.258129,0.259906,3.912725
min,1.000000,1.000000,1.000000,1.000000,67.012000,2022.000000,-2403.895000,-274.564000,0.000000,0.000000,0.012821,0.012821,-18.000000
25%,15.000000,1.000000,7.000000,5.000000,82.021000,2023.000000,-7.253000,-51.054500,0.000000,0.000000,0.210526,0.172414,-2.000000
50%,30.000000,2.000000,13.000000,10.000000,91.167000,2024.000000,-0.027000,-21.678000,0.000000,0.000000,0.421053,0.333333,0.000000
75%,45.000000,3.000000,20.000000,14.000000,99.356000,2025.000000,5.705000,-3.725500,1.000000,1.000000,0.631579,0.562500,2.000000
max,78.000000,8.000000,78.000000,20.000000,2526.253000,2025.000000,2433.472000,2412.431000,1.000000,1.000000,1.000000,1.000000,18.000000


In [ ]:
# SEPARATING THE COLUMNS FOR EASIRE ACCESS

NUMERIC = ['Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change']

CATEGORICAL = ['Driver', 'Compound', 'Race']

TARGET = 'PitNextLap'